In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder.getOrCreate()

SILVER_DB = "petroleum_silver"
GOLD_DB   = "petroleum_gold"
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

df_silver = spark.table(f"{SILVER_DB}.petroleum_weekly")
print(f"Silver rows loaded: {df_silver.count()}")


In [0]:
df_gold_weekly = (
    df_silver
    .select(
        "date", "year", "month", "week", "era",
        "wti_price", "brent_price", "wti_brent_spread",
        "wti_wow_pct", "wti_4wk_avg",
        "brent_wow_pct",
        "us_production_mbpd",
        "crude_stocks_mb", "stocks_wow_chg",
        "product_supplied_mbpd",
        "gasoline_price_gal", "diesel_price_gal",
        "gas_52wk_avg",
        "conflict_flag"
    )
    .orderBy("date")
)

df_gold_weekly.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{GOLD_DB}.petroleum_weekly_fact")
print(f"Gold weekly fact: {df_gold_weekly.count()} rows")


In [0]:
# Era average table — drives the comparison bar chart in Tableau
df_era_avg = (
    df_silver
    .groupBy("era")
    .agg(
        F.round(F.avg("wti_price"),          2).alias("avg_wti_price"),
        F.round(F.avg("brent_price"),         2).alias("avg_brent_price"),
        F.round(F.avg("gasoline_price_gal"),  3).alias("avg_gasoline_price"),
        F.round(F.avg("diesel_price_gal"),    3).alias("avg_diesel_price"),
        F.round(F.max("wti_price"),           2).alias("max_wti_price"),
        F.round(F.min("wti_price"),           2).alias("min_wti_price"),
        F.count("*").alias("week_count")
    )
    .orderBy("era")
)

df_era_avg.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{GOLD_DB}.era_averages")
print("Era averages table:")
df_era_avg.show(truncate=False)


In [0]:
def safe_round(value, decimals):
    """Return rounded value or 'N/A' if None"""
    return str(round(value, decimals)) if value is not None else "N/A"

latest  = df_silver.orderBy(F.col("date").desc()).limit(1).collect()[0]
jan2026 = df_silver.filter(
    (F.year("date") == 2026) & (F.month("date") == 1)
).orderBy("date").limit(1).collect()[0]

all_time_high = df_silver.agg(F.max("wti_price")).collect()[0][0]
covid_low     = df_silver.filter(F.year("date") == 2020) \
                         .agg(F.min("wti_price")).collect()[0][0]

kpis = [
    {"kpi": "Current WTI Price ($/bbl)",
     "value": safe_round(latest["wti_price"], 2),       "unit": "$/barrel"},

    {"kpi": "Current Brent Price ($/bbl)",
     "value": safe_round(latest["brent_price"], 2),     "unit": "$/barrel"},

    {"kpi": "WTI % Change Since Jan 2026",
     "value": safe_round((latest["wti_price"] - jan2026["wti_price"]) / jan2026["wti_price"] * 100, 1) if latest["wti_price"] is not None and jan2026["wti_price"] is not None else "N/A",
     "unit": "%"},

    {"kpi": "WTI vs Brent Spread",
     "value": safe_round(latest["wti_brent_spread"], 2), "unit": "$/barrel"},

    {"kpi": "All-Time WTI High (full history)",
     "value": safe_round(all_time_high, 2),               "unit": "$/barrel"},

    {"kpi": "2020 COVID WTI Low",
     "value": safe_round(covid_low, 2),                   "unit": "$/barrel"},

    {"kpi": "Current Gasoline Retail Price",
     "value": safe_round(latest["gasoline_price_gal"], 3), "unit": "$/gallon"},

    {"kpi": "Current Diesel Retail Price",
     "value": safe_round(latest["diesel_price_gal"], 3),  "unit": "$/gallon"},

    {"kpi": "US Crude Stocks (latest)",
     "value": safe_round(latest["crude_stocks_mb"], 1),   "unit": "million barrels"},
]

df_kpis = spark.createDataFrame(kpis)
df_kpis.write.format("delta").mode("overwrite") \
    .saveAsTable(f"{GOLD_DB}.headline_kpis")
print("\n=== HEADLINE KPIs ===")
df_kpis.show(truncate=False)

In [0]:
# Export via display + download ──────────

# Write as managed Delta tables in your catalog (this always works)
df_gold_weekly.write.format("delta").mode("overwrite") \
    .saveAsTable("petroleum_gold.petroleum_weekly_fact")

df_era_avg.write.format("delta").mode("overwrite") \
    .saveAsTable("petroleum_gold.era_averages")

print("Gold tables written successfully.")

In [0]:
%sql
SHOW CATALOGS;


In [0]:
%sql
--SELECT CURRENT_METASTORE();
CREATE CATALOG IF NOT EXISTS petroleum_gold;

USE CATALOG petroleum_gold



In [0]:
#Export to Unity Catalog Volume (correct UC approach)
spark.sql(" USE CATALOG workspace")


spark.sql("CREATE SCHEMA IF NOT EXISTS petroleum_gold.exports")
spark.sql("CREATE VOLUME IF NOT EXISTS petroleum_gold.exports.csv_exports")

df_gold_weekly.coalesce(1).write.option("header","true").mode("overwrite").csv("/Volumes/petroleum_gold/exports/csv_exports/weekly")
df_era_avg.coalesce(1).write.option("header","true").mode("overwrite").csv("/Volumes/petroleum_gold/exports/csv_exports/era_avg")

print("Exported to /Volumes/petroleum_gold/exports/csv_exports/")
print("Download: Catalog → petroleum_gold → exports → csv_exports → Browse files")

In [0]:
# Simply query the table you already wrote
display(spark.sql("SELECT * FROM petroleum_gold.petroleum_weekly_fact"))

In [0]:
display(spark.sql("SELECT * FROM petroleum_gold.era_averages"))